In [ ]:
import os
import json
import numpy as np

RUN_ROOT = os.path.join(os.getcwd(), "runs_fixed_pool_rerank")

# SET YOUR UNIQUE TIMED FILE PATHS AFTER A NEW TRAINING:
RUNS = {
    "Llama": os.path.join(RUN_ROOT, "exp1_llama_20260131_140548"),
    "Qwen": os.path.join(RUN_ROOT, "exp1_qwen_20260131_143032"),
    "GPT": os.path.join(RUN_ROOT, "exp1_gpt_20260131_154255"),
}


In [17]:
def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def read_jsonl(path):
    out = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            out.append(json.loads(line))
    return out

def mean(rows, key):
    return sum(r[key] for r in rows) / len(rows) if rows else 0.0

def assert_exists(path):
    if not os.path.exists(path):
        raise RuntimeError("Missing file: " + path)


In [18]:
ART = {}

for name, run_dir in RUNS.items():
    meta_path = os.path.join(run_dir, "meta.json")
    rank_path = os.path.join(run_dir, "rank_rows.jsonl")
    flat_path = os.path.join(run_dir, "results_by_keep_flat.jsonl")
    sem_path = os.path.join(run_dir, "semantic_bootstrap.json")

    assert_exists(meta_path)
    assert_exists(rank_path)
    assert_exists(flat_path)
    assert_exists(sem_path)

    meta = read_json(meta_path)
    rank_rows = read_jsonl(rank_path)
    flat = read_jsonl(flat_path)
    sem_boot = read_json(sem_path)

    ART[name] = {
        "run_dir": run_dir,
        "meta": meta,
        "rank_rows": rank_rows,
        "flat": flat,
        "sem_boot": sem_boot,
    }

    print(name, "loaded")
    print("  model_name", meta.get("model_name"))
    print("  n_examples_used", meta.get("n_examples_used"))
    print()


Llama loaded
  model_name llama3.1:8b-instruct-q4_K_M
  n_examples_used 345

Qwen loaded
  model_name qwen2.5:7b-instruct-q4_K_M
  n_examples_used 345

GPT loaded
  model_name gpt-5-mini
  n_examples_used 345



In [19]:
def agreement_stats(rank_rows, flat):
    tau_llm_bm25 = mean(rank_rows, "tau_llm_bm25")
    tau_llm_mmr = mean(rank_rows, "tau_llm_mmr")
    tau_llm_rand = mean(rank_rows, "tau_llm_rand")

    rows_k3 = [r for r in flat if int(r["keep"]) == 3]
    if not rows_k3:
        raise RuntimeError("No keep=3 rows found")

    j_llm_bm25 = mean(rows_k3, "jacc_llm_bm25")
    j_llm_mmr = mean(rows_k3, "jacc_llm_mmr")
    j_llm_rand = mean(rows_k3, "jacc_llm_rand")

    return {
        "tau_llm_bm25": tau_llm_bm25,
        "tau_llm_mmr": tau_llm_mmr,
        "tau_llm_rand": tau_llm_rand,
        "jacc3_llm_bm25": j_llm_bm25,
        "jacc3_llm_mmr": j_llm_mmr,
        "jacc3_llm_rand": j_llm_rand,
    }

AGREE = {}
for name in ["Llama", "Qwen", "GPT"]:
    AGREE[name] = agreement_stats(ART[name]["rank_rows"], ART[name]["flat"])


In [20]:
def fmt(x, nd=2):
    return f"{x:.{nd}f}"

print("Agreement table rows")
for name in ["Llama", "Qwen", "GPT"]:
    s = AGREE[name]
    row = (
        f"\\texttt{{{name}}} & "
        f"{fmt(s['tau_llm_bm25'], 2)} & {fmt(s['tau_llm_mmr'], 2)} & "
        f"{fmt(s['jacc3_llm_bm25'], 2)} & {fmt(s['jacc3_llm_mmr'], 2)} & {fmt(s['jacc3_llm_rand'], 2)} \\\\"
    )
    print(row)


Agreement table rows
\texttt{Llama} & 0.19 & 0.19 & 0.37 & 0.37 & 0.27 \\
\texttt{Qwen} & 0.33 & 0.32 & 0.42 & 0.41 & 0.25 \\
\texttt{GPT} & 0.41 & 0.39 & 0.48 & 0.47 & 0.27 \\


In [21]:
METHOD_PREFIX = {
    "BM25": "bm25",
    "MMR": "mmr",
    "LLM": "llm",
    "Random": "rand",
}

COMPARISONS = [
    ("MMR", "LLM"),
    ("BM25", "LLM"),
    ("LLM", "Random"),
]

LEX_METRICS = ["cov_cont", "red_cont", "sum_rec"]

def paired_bootstrap_ci(a, b, n_boot=10000, seed=12345, alpha=0.05):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n = len(a)
    rng = np.random.default_rng(seed)

    obs = float(np.mean(a - b))
    idx = rng.integers(0, n, size=(n_boot, n))
    diffs = (a[idx] - b[idx]).mean(axis=1)

    lo, hi = np.quantile(diffs, [alpha / 2, 1 - alpha / 2])
    p_pos = float(np.mean(diffs > 0))
    return obs, float(lo), float(hi), p_pos

def get_metric_array(rows, method_label, metric_key):
    prefix = METHOD_PREFIX[method_label]
    col = f"{prefix}_{metric_key}"
    if col not in rows[0]:
        raise RuntimeError("Missing column " + col)
    return np.array([r[col] for r in rows], dtype=float)

def bootstrap_for_k(flat_rows, k_focus, metric_keys, n_boot=10000, seed=12345):
    subset = [r for r in flat_rows if int(r["keep"]) == int(k_focus)]
    if not subset:
        raise RuntimeError("No rows for keep " + str(k_focus))

    out = {}
    for metric_key in metric_keys:
        out[metric_key] = {}
        for A, B in COMPARISONS:
            a = get_metric_array(subset, A, metric_key)
            b = get_metric_array(subset, B, metric_key)
            obs, lo, hi, ppos = paired_bootstrap_ci(a, b, n_boot=n_boot, seed=seed, alpha=0.05)
            out[metric_key][f"{A}-{B}"] = {
                "mean_delta": obs,
                "lo": lo,
                "hi": hi,
                "p_pos": ppos,
            }
    return out

LEX_BOOT = {}
for name in ["Llama", "Qwen", "GPT"]:
    flat = ART[name]["flat"]
    LEX_BOOT[name] = {
        "K3": bootstrap_for_k(flat, 3, LEX_METRICS),
        "K5": bootstrap_for_k(flat, 5, LEX_METRICS),
    }


In [22]:
def fmt_ci(mean_delta, lo, hi):
    return f"{mean_delta:+0.3f} [{lo:+0.3f},{hi:+0.3f}]"

print("Bootstrap deltas")
for name in ["Llama", "Qwen", "GPT"]:
    b = LEX_BOOT[name]
    print(name)

    for metric_key in ["cov_cont", "red_cont", "sum_rec"]:
        k3 = b["K3"][metric_key]
        k5 = b["K5"][metric_key]

        line = (
            f"{metric_key}  "
            f"K3 {fmt_ci(k3['MMR-LLM']['mean_delta'], k3['MMR-LLM']['lo'], k3['MMR-LLM']['hi'])}  "
            f"{fmt_ci(k3['BM25-LLM']['mean_delta'], k3['BM25-LLM']['lo'], k3['BM25-LLM']['hi'])}  "
            f"{fmt_ci(k3['LLM-Random']['mean_delta'], k3['LLM-Random']['lo'], k3['LLM-Random']['hi'])}  "
            f"K5 {fmt_ci(k5['MMR-LLM']['mean_delta'], k5['MMR-LLM']['lo'], k5['MMR-LLM']['hi'])}  "
            f"{fmt_ci(k5['BM25-LLM']['mean_delta'], k5['BM25-LLM']['lo'], k5['BM25-LLM']['hi'])}  "
            f"{fmt_ci(k5['LLM-Random']['mean_delta'], k5['LLM-Random']['lo'], k5['LLM-Random']['hi'])}"
        )
        print(line)

    print()


Bootstrap deltas
Llama
cov_cont  K3 +0.091 [+0.077,+0.106]  +0.090 [+0.076,+0.106]  +0.056 [+0.037,+0.076]  K5 +0.045 [+0.036,+0.056]  +0.045 [+0.036,+0.056]  +0.033 [+0.019,+0.047]
red_cont  K3 -0.000 [-0.009,+0.007]  +0.004 [-0.003,+0.012]  +0.015 [+0.004,+0.025]  K5 +0.003 [-0.002,+0.008]  +0.009 [+0.004,+0.014]  +0.010 [+0.004,+0.015]
sum_rec  K3 +0.008 [+0.002,+0.014]  +0.009 [+0.003,+0.015]  +0.017 [+0.010,+0.025]  K5 +0.008 [+0.003,+0.014]  +0.007 [+0.002,+0.013]  +0.014 [+0.008,+0.020]

Qwen
cov_cont  K3 +0.057 [+0.048,+0.067]  +0.057 [+0.048,+0.067]  +0.089 [+0.071,+0.108]  K5 +0.025 [+0.020,+0.031]  +0.025 [+0.020,+0.031]  +0.053 [+0.041,+0.066]
red_cont  K3 -0.007 [-0.013,-0.001]  -0.002 [-0.008,+0.004]  +0.022 [+0.009,+0.034]  K5 -0.003 [-0.006,+0.001]  +0.004 [+0.000,+0.007]  +0.015 [+0.010,+0.020]
sum_rec  K3 +0.003 [-0.002,+0.008]  +0.004 [-0.001,+0.009]  +0.022 [+0.015,+0.030]  K5 +0.003 [-0.001,+0.008]  +0.002 [-0.003,+0.007]  +0.019 [+0.013,+0.024]

GPT
cov_cont  K3 +

In [23]:
print("Semantic bootstrap deltas")
for name in ["Llama", "Qwen", "GPT"]:
    b = ART[name]["sem_boot"]
    print(name)

    for metric_key in ["semred", "semcov"]:
        k3 = b["K3"][metric_key]
        k5 = b["K5"][metric_key]

        line = (
            f"{metric_key}  "
            f"K3 {fmt_ci(k3['MMR-LLM']['mean_delta'], k3['MMR-LLM']['lo'], k3['MMR-LLM']['hi'])}  "
            f"{fmt_ci(k3['BM25-LLM']['mean_delta'], k3['BM25-LLM']['lo'], k3['BM25-LLM']['hi'])}  "
            f"{fmt_ci(k3['LLM-Random']['mean_delta'], k3['LLM-Random']['lo'], k3['LLM-Random']['hi'])}  "
            f"K5 {fmt_ci(k5['MMR-LLM']['mean_delta'], k5['MMR-LLM']['lo'], k5['MMR-LLM']['hi'])}  "
            f"{fmt_ci(k5['BM25-LLM']['mean_delta'], k5['BM25-LLM']['lo'], k5['BM25-LLM']['hi'])}  "
            f"{fmt_ci(k5['LLM-Random']['mean_delta'], k5['LLM-Random']['lo'], k5['LLM-Random']['hi'])}"
        )
        print(line)

    print()


Semantic bootstrap deltas
Llama
semred  K3 +0.006 [-0.011,+0.023]  +0.016 [+0.000,+0.032]  +0.112 [+0.090,+0.134]  K5 +0.019 [+0.007,+0.030]  +0.029 [+0.018,+0.040]  +0.070 [+0.056,+0.083]
semcov  K3 +0.007 [+0.002,+0.012]  +0.007 [+0.002,+0.012]  +0.014 [+0.008,+0.020]  K5 +0.005 [+0.001,+0.010]  +0.005 [+0.001,+0.009]  +0.004 [-0.001,+0.008]

Qwen
semred  K3 -0.023 [-0.036,-0.009]  -0.012 [-0.025,+0.001]  +0.140 [+0.118,+0.162]  K5 -0.003 [-0.012,+0.007]  +0.008 [-0.002,+0.017]  +0.091 [+0.079,+0.102]
semcov  K3 +0.001 [-0.002,+0.004]  +0.001 [-0.002,+0.004]  +0.019 [+0.013,+0.026]  K5 -0.000 [-0.002,+0.002]  -0.001 [-0.003,+0.001]  +0.009 [+0.006,+0.012]

GPT
semred  K3 -0.044 [-0.058,-0.031]  -0.034 [-0.046,-0.021]  +0.162 [+0.141,+0.182]  K5 -0.033 [-0.042,-0.024]  -0.023 [-0.030,-0.015]  +0.121 [+0.109,+0.133]
semcov  K3 +0.002 [-0.001,+0.005]  +0.002 [-0.001,+0.006]  +0.018 [+0.012,+0.025]  K5 +0.000 [-0.002,+0.003]  -0.000 [-0.002,+0.002]  +0.009 [+0.006,+0.012]

